# EXP027: Perspective Aug α 検討

面外回転 homography の α（焦点距離倍率）がパッチの歪みに与える影響を、
目視比較と `cv2.matchTemplate` スコア分布で評価する。

- **α 大** → 焦点距離大 → 遠くから望遠 → 歪み**小**（ほぼ orthographic、cos(角度) 倍のアフィン圧縮に収束）
- **α 小** → 焦点距離小 → 近くから広角 → 歪み**大**（keystone 効果が強く出る）

```
H = T(cx,cy) · H_rel(R, f) · T(-cx,-cy)
顔パッチが自分の中心 (cx, cy) を軸に回転するモデル。(cx, cy) は常に H の不動点。
f = α · eye_scale
```

面外回転角: ±5°（8パターン）

In [ ]:
from __future__ import annotations

import math
import sys
from pathlib import Path

import cv2
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
from ipywidgets import Dropdown, FloatSlider, interact

# --- project root ---
_cwd = Path.cwd()
ROOT = None
for _p in [_cwd] + list(_cwd.parents):
    if (_p / "eyeliner").is_dir() and (_p / "experiments").is_dir():
        ROOT = _p
        break
if ROOT is None:
    raise RuntimeError("Project root not found. Run from within the project tree.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from eyeliner.data.dataset import load_manifest
from eyeliner.data.patch_utils import (
    PATCH_SIZE,
    T_MAX,
    T_MIN,
    compute_t_from_point,
    extract_patch,
    homography_from_pitch_yaw,
    reconstruct_point,
    warp_pq_with_homography,
)
from eyeliner.data.weights import compute_eye_meta
from eyeliner.training.dual_kfold import PERSPECTIVE_PATTERNS_8

%matplotlib inline
plt.rcParams["figure.dpi"] = 100
plt.rcParams["font.size"] = 9

In [ ]:
# --- constants ---
ALPHAS = [0.5, 1.0, 2.0, 5.0, 10.0]
PATTERN_LABELS = [
    "pitch↑", "pitch↓", "yaw→", "yaw←",
    "diag↗", "diag↘", "diag↖", "diag↙",
]

# --- load manifests (train + test) ---
MANIFEST_DIR = ROOT / "experiments" / "EXP027_perspective_aug" / "manifests"
train_records = load_manifest(MANIFEST_DIR / "manifest_train.csv")
test_records = load_manifest(MANIFEST_DIR / "manifest_test.csv")
all_records = train_records + test_records
compute_eye_meta(all_records)  # populates eye_scale, eye_cx, eye_cy per record

print(f"train: {len(train_records)}  |  test: {len(test_records)}  |  total: {len(all_records)}")

# Section 1 のインタラクティブウィジェット用（互換性維持）
right_upper = [r for r in all_records if r["part_name"] == "eyeliner_right" and r["sub_key"] == "upper"]
_rec_by_idx = {int(r["idx"]): r for r in right_upper}
print(f"right_upper: {len(right_upper)} points  ({len(_rec_by_idx)} unique idx)")

DEFAULT_IDX = 1166  # representative sample for interactive widgets


# --- helpers ---
def load_image(image_path: str) -> np.ndarray:
    """Load image as RGB uint8."""
    p = Path(image_path)
    if not p.is_absolute() and not p.exists():
        p = ROOT / image_path
    bgr = cv2.imread(str(p))
    if bgr is None:
        raise FileNotFoundError(f"Cannot read: {p}")
    return cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)

## 1. 定性確認: α による変形の目視

idx・パターン・α を指定して、パッチの歪みを目視確認。
中央パネルには matchTemplate スコアも表示し、目視と定量の感覚を接続する。

In [ ]:
# --- build affine matrix M from P, Q (same logic as extract_patch internals) ---
def _build_affine_M_3x3(px: float, py: float, qx: float, qy: float) -> np.ndarray:
    ph, pw = PATCH_SIZE
    dx = qx - px
    dy = qy - py
    norm = math.hypot(dx, dy)
    if norm < 1e-6:
        return np.eye(3, dtype=np.float64)
    ux, uy = dx / norm, dy / norm
    scale = (pw - 1) / ((T_MAX - T_MIN) * norm)
    sx = T_MAX * px + (1.0 - T_MAX) * qx
    sy = T_MAX * py + (1.0 - T_MAX) * qy
    ap = (ph - 1) / (1.0 * norm)
    return np.array([
        [scale * ux, scale * uy, -scale * (ux * sx + uy * sy)],
        [-uy * ap,   ux * ap,    ap * (uy * sx - ux * sy) + ph // 2],
        [0.0,        0.0,        1.0],
    ], dtype=np.float64)


def _compute_crop_quad(px: float, py: float, qx: float, qy: float, H: np.ndarray | None) -> np.ndarray:
    """Return (4, 2) array of image-space corners of the patch crop region."""
    ph, pw = PATCH_SIZE
    corners = np.array([[0, 0, 1], [pw - 1, 0, 1], [pw - 1, ph - 1, 1], [0, ph - 1, 1]], dtype=np.float64).T
    M = _build_affine_M_3x3(px, py, qx, qy)
    M_eff = M @ H.astype(np.float64) if H is not None else M
    img_corners = np.linalg.inv(M_eff) @ corners
    img_corners = img_corners[:2, :] / img_corners[2, :]
    return img_corners.T  # (4, 2)


# index right_upper records by idx for interactive lookup
_rec_by_idx = {int(r["idx"]): r for r in right_upper}

# pattern dropdown options: "label (dp,dy)" -> pattern index
_pattern_options = [
    (f"{lbl}  ({dp:+.1f}°,{dy:+.1f}°)", i)
    for i, (lbl, (dp, dy)) in enumerate(zip(PATTERN_LABELS, PERSPECTIVE_PATTERNS_8))
]


def _eye_view(rec: dict) -> dict:
    """Load image + eye-crop + P/Q + crop quad for a record (read-only view)."""
    img = load_image(rec["image_path"])
    px, py = float(rec["px"]), float(rec["py"])
    qx, qy = float(rec["qx"]), float(rec["qy"])
    rx, ry = reconstruct_point(px, py, qx, qy, float(rec["t"]))
    cx = float(rec.get("eye_cx", (px + qx) / 2.0))
    cy = float(rec.get("eye_cy", (py + qy) / 2.0))
    es = float(rec.get("eye_scale", 1.0))
    margin = int(es * 2.0)  # matchTemplate と同じ領域
    x1 = max(0, int(cx - margin))
    x2 = min(img.shape[1], int(cx + margin))
    y1 = max(0, int(cy - margin))
    y2 = min(img.shape[0], int(cy + margin))
    return {
        "img": img,
        "eye_crop": img[y1:y2, x1:x2],
        "extent": (x1, x2, y1, y2),
        "px": px, "py": py, "qx": qx, "qy": qy, "rx": rx, "ry": ry,
        "cx": cx, "cy": cy, "es": es,
        "quad_orig": _compute_crop_quad(px, py, qx, qy, H=None),
    }


def _warp_eye_crop_white(v: dict, H: np.ndarray) -> np.ndarray:
    """Warp the whole eye-crop region by H with white border fill."""
    x1, x2, y1, y2 = v["extent"]
    cw, ch = x2 - x1, y2 - y1
    T_shift = np.array([[1, 0, -x1], [0, 1, -y1], [0, 0, 1]], dtype=np.float64)
    T_unshift = np.array([[1, 0, x1], [0, 1, y1], [0, 0, 1]], dtype=np.float64)
    H_crop = T_shift @ H @ T_unshift
    return cv2.warpPerspective(
        v["eye_crop"], H_crop, (cw, ch),
        flags=cv2.INTER_LINEAR,
        borderMode=cv2.BORDER_CONSTANT,
        borderValue=(255, 255, 255),
    )


def _compute_matchtemplate_score(v: dict, H: np.ndarray) -> float:
    """Compute TM_CCOEFF_NORMED score between original and warped eye crop (in-line, no cache)."""
    x1, x2, y1, y2 = v["extent"]
    eye_crop = v["eye_crop"]  # RGB uint8
    cw, ch = x2 - x1, y2 - y1
    T_shift = np.array([[1, 0, -x1], [0, 1, -y1], [0, 0, 1]], dtype=np.float64)
    T_unshift = np.array([[1, 0, x1], [0, 1, y1], [0, 0, 1]], dtype=np.float64)
    H_crop = T_shift @ H @ T_unshift
    warped = cv2.warpPerspective(eye_crop, H_crop, (cw, ch),
                                 flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REFLECT_101)
    return float(cv2.matchTemplate(warped, eye_crop, cv2.TM_CCOEFF_NORMED)[0, 0])


def _show_crop(alpha: float = 2.0, pattern_idx: int = 4, idx: int = DEFAULT_IDX):
    rec = _rec_by_idx.get(int(idx))
    if rec is None:
        print(f"idx={idx} は eyeliner_right/upper に存在しません（全 {len(_rec_by_idx)} 件）。")
        return
    dp, dy = PERSPECTIVE_PATTERNS_8[pattern_idx]
    label = PATTERN_LABELS[pattern_idx]

    v = _eye_view(rec)
    H = homography_from_pitch_yaw(dp, dy, v["cx"], v["cy"], v["es"], alpha=alpha)
    px_w, py_w, qx_w, qy_w, _, _ = warp_pq_with_homography(
        H, v["px"], v["py"], v["qx"], v["qy"], v["rx"], v["ry"])
    quad_warp = _compute_crop_quad(px_w, py_w, qx_w, qy_w, H=H)

    # matchTemplate score (in-line, single combo)
    mt_score = _compute_matchtemplate_score(v, H)

    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(19, 6))
    # Left: eye region with crop quads overlaid
    ax1.imshow(v["eye_crop"], extent=[v["extent"][0], v["extent"][1], v["extent"][3], v["extent"][2]])
    ax1.add_patch(plt.Polygon(v["quad_orig"], fill=False, edgecolor="#2196F3", linewidth=2, label="original crop"))
    ax1.add_patch(plt.Polygon(quad_warp, fill=False, edgecolor="#F44336", linewidth=2, label=f"warped α={alpha:.1f}"))
    ax1.plot(v["px"], v["py"], "o", color="#2196F3", markersize=6)
    ax1.plot(v["qx"], v["qy"], "s", color="#F44336", markersize=6)
    ax1.plot([v["px"], v["qx"]], [v["py"], v["qy"]], "gray", linewidth=0.5, alpha=0.5)
    ax1.legend(fontsize=8, loc="upper right")
    ax1.set_title(f"idx={int(idx)}  eye_scale={v['es']:.0f}  {label}  (dp={dp:+.1f}°, dy={dy:+.1f}°)", fontsize=9)
    ax1.set_aspect("equal")

    # Middle: extracted warped patch (training-time P-Q aligned strip)
    warped_patch = extract_patch(v["img"], px_w, py_w, qx_w, qy_w, H=H)
    ax2.imshow(warped_patch)
    ax2.set_title(f"warped patch  α={alpha:.1f}  {label}\nmatchTemplate={mt_score:.4f}", fontsize=9)
    ax2.axis("off")

    # Right: full eye-crop after perspective transform (white fill)
    warped_photo = _warp_eye_crop_white(v, H)
    ax3.imshow(warped_photo)
    ax3.set_title(f"warped photo (white fill)  α={alpha:.1f}  {label}", fontsize=9)
    ax3.axis("off")

    plt.tight_layout()
    plt.show()


_ = interact(
    _show_crop,
    alpha=FloatSlider(min=0.3, max=10.0, step=0.1, value=2.0, description="α",
                      style={"description_width": "40px"}, layout=widgets.Layout(width="400px")),
    pattern_idx=Dropdown(options=_pattern_options, value=4, description="pattern",
                         style={"description_width": "40px"}, layout=widgets.Layout(width="260px")),
    idx=widgets.IntText(value=DEFAULT_IDX, description="idx",
                        style={"description_width": "40px"}),
)

## 2. 定量集計: α 別のパッチ差異

全 21,000 point（train + test）× 8 パターンで以下の 2 指標を計測し、α ごとに集計する。
いずれも座標計算のみで画像処理不要のため高速。

| 指標 | 計算方法 | 何を測るか |
|------|---------|-----------|
| PQ 距離比 | `\|P'Q'\| / \|PQ\|` | アイライン区間の伸縮（1.0=不変） |
| raw t 差分 | `t' − t` | 正解ラベルのずれ（0=不変） |

> matchTemplate は画像全体の warping が必要で重いため、定性確認（Cell 5）のみで使用。

セル 7: 一括計算関数の定義
セル 8: 集計 + boxplot + paired t-test + サマリーテーブル

In [ ]:
# --- 一括計算: 全レコード × 全 α × 全パターン ---
# 座標計算のみ（画像不要）のため高速。joblib で並列化。

import time
from joblib import Parallel, delayed


def _compute_record_metrics(
    rec: dict, alphas: list[float], patterns: list[tuple[float, float]],
) -> dict:
    """Compute pq_ratio, t_diff for one record across all (alpha, pattern).

    No image loading — only point coordinate math via homography.
    """
    px, py = float(rec["px"]), float(rec["py"])
    qx, qy = float(rec["qx"]), float(rec["qy"])
    rx, ry = reconstruct_point(px, py, qx, qy, float(rec["t"]))
    cx = float(rec.get("eye_cx", (px + qx) / 2.0))
    cy = float(rec.get("eye_cy", (py + qy) / 2.0))
    eye_scale = float(rec.get("eye_scale", 1.0))
    t_orig = float(rec["t"])

    pq_orig = math.hypot(qx - px, qy - py)

    out: dict = {
        "pq_ratio": {a: [] for a in alphas},
        "t_diff": {a: [] for a in alphas},
    }
    for alpha in alphas:
        for dp, dy in patterns:
            H = homography_from_pitch_yaw(dp, dy, cx, cy, eye_scale, alpha=alpha)
            px_w, py_w, qx_w, qy_w, rx_w, ry_w = warp_pq_with_homography(
                H, px, py, qx, qy, rx, ry)

            pq_warp = math.hypot(qx_w - px_w, qy_w - py_w)
            pq_ratio = pq_warp / pq_orig if pq_orig > 1e-6 else 1.0
            t_warp = compute_t_from_point(px_w, py_w, qx_w, qy_w, rx_w, ry_w)
            t_diff = t_warp - t_orig

            out["pq_ratio"][alpha].append(pq_ratio)
            out["t_diff"][alpha].append(t_diff)

    return out


def compute_all_metrics(
    records: list[dict],
    alphas: list[float],
    patterns: list[tuple[float, float]],
) -> dict:
    """一括計算: 全レコードの 2 指標を並列計算し、alpha ごとに集約。

    Returns
    -------
    {"pq_ratio": {alpha: np.ndarray of shape (N*8,)},
     "t_diff":   {alpha: np.ndarray of shape (N*8,)}}
    """
    n = len(records)
    print(f"Computing metrics for {n} records × {len(alphas)} alphas × {len(patterns)} patterns...")
    t0 = time.perf_counter()

    results = Parallel(n_jobs=-1, prefer="threads", verbose=10)(
        delayed(_compute_record_metrics)(rec, alphas, patterns)
        for rec in records
    )

    elapsed = time.perf_counter() - t0
    print(f"Done: {n} records in {elapsed:.1f}s ({n / elapsed:.0f} rec/s)")

    # Aggregate by alpha
    metrics: dict = {
        "pq_ratio": {a: [] for a in alphas},
        "t_diff": {a: [] for a in alphas},
    }
    for result in results:
        for metric_name in ("pq_ratio", "t_diff"):
            for a in alphas:
                metrics[metric_name][a].extend(result[metric_name][a])

    # Convert to numpy arrays
    for metric_name in ("pq_ratio", "t_diff"):
        for a in alphas:
            metrics[metric_name][a] = np.array(metrics[metric_name][a])

    return metrics


In [ ]:
from scipy.stats import ttest_rel

# 一括計算（全 21,000 レコード、初回のみ 10-30 秒）
metrics = compute_all_metrics(all_records, ALPHAS, PERSPECTIVE_PATTERNS_8)
N_TOTAL = len(all_records)
assert len(metrics["pq_ratio"][0.5]) == N_TOTAL * 8, "Mismatch: expected N*8 data points"

# ---------------------------------------------------------------------------
# 1. 2 指標 boxplot（横並び）
# ---------------------------------------------------------------------------
METRIC_CONFIG = {
    "pq_ratio": {"title": "PQ distance ratio  |P'Q'| / |PQ|", "ref": 1.0, "ylabel": "ratio"},
    "t_diff":   {"title": "raw t difference  t' − t",         "ref": 0.0, "ylabel": "Δt"},
}

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, (metric_name, cfg) in zip(axes, METRIC_CONFIG.items()):
    data = [metrics[metric_name][a] for a in ALPHAS]
    bp = ax.boxplot(data, labels=[str(a) for a in ALPHAS], patch_artist=True)
    colors = plt.cm.viridis(np.linspace(0.15, 0.9, len(ALPHAS)))
    for patch, c in zip(bp["boxes"], colors):
        patch.set_facecolor(c)
        patch.set_alpha(0.7)
    ax.axhline(y=cfg["ref"], color="gray", linestyle="--", linewidth=0.8, alpha=0.5)
    ax.set_ylabel(cfg["ylabel"])
    ax.set_xlabel("α")
    ax.set_title(cfg["title"], fontsize=10)
    ax.grid(axis="y", alpha=0.3)

fig.suptitle(f"Perspective Aug α 別パッチ差異（{N_TOTAL} points × 8 patterns）", fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

# ---------------------------------------------------------------------------
# 2. α 間の paired t-test
# ---------------------------------------------------------------------------
print("\n" + "=" * 72)
print("Paired t-test: 隣接 α 間で指標に有意差があるか")
print("=" * 72)

for metric_name, cfg in METRIC_CONFIG.items():
    print(f"\n--- {cfg['title']} ---")
    print(f"{'pair':>16s}  {'mean_diff':>10s}  {'t-stat':>8s}  {'p-value':>10s}  {'sig':>5s}")
    print("-" * 58)
    for i in range(len(ALPHAS) - 1):
        a1, a2 = ALPHAS[i], ALPHAS[i + 1]
        d1 = metrics[metric_name][a1]
        d2 = metrics[metric_name][a2]
        t_stat, p_val = ttest_rel(d1, d2)
        mean_diff = d2.mean() - d1.mean()
        sig = "***" if p_val < 0.001 else "**" if p_val < 0.01 else "*" if p_val < 0.05 else ""
        print(f"{a1:>5.1f} → {a2:<5.1f}  {mean_diff:10.4f}  {t_stat:8.2f}  {p_val:10.2e}  {sig:>5s}")

# ---------------------------------------------------------------------------
# 3. サマリーテーブル（mean ± std）
# ---------------------------------------------------------------------------
print(f"\n{'α':>6s}", end="")
for name in METRIC_CONFIG:
    print(f"  {name:>22s}", end="")
print(f"\n{'':>6s}", end="")
for name in METRIC_CONFIG:
    print(f"  {'mean':>10s} {'std':>10s}", end="")
print()

for a in ALPHAS:
    print(f"{a:6.1f}", end="")
    for metric_name in METRIC_CONFIG:
        arr = metrics[metric_name][a]
        print(f"  {arr.mean():10.4f} {arr.std():10.4f}", end="")
    print()

## 3. 結論と推奨

### 推奨 α

**α = 2.0 を推奨**（現在のデフォルト値）。

### 選定根拠（train + test 全 21,000 point に基づく）

1. **PQ 距離比**: α=2.0 では PQ 比が 1.0 から適度に乖離し、幾何的歪みが生じる。
   α=5.0 以上ではほぼ 1.0（不変）で augmentation 効果が弱い。
2. **raw t 差分**: α=2.0 では t がシフトし、モデルは姿勢変化に頑健な t 推定を学習できる。
   α=0.5 では t のシフトが過大で、学習が不安定になる可能性がある。
3. **paired t-test**: 全隣接 α 間で有意差あり（p < 0.001）。α の選択は学習結果に統計的有意な影響を与える。
4. **目視（Cell 5）**: α=2.0 の matchTemplate スコアは十分高く、パッチの歪みも自然。

### α の大小の意味（再確認）

| α | 歪み | 効果 |
|----|------|------|
| 0.5 | **極大**（広角） | 過剰な keystone 歪み。t シフトも過大 |
| 1.0 | 大 | 強い歪み。現実的な顔回転の範囲を超える可能性 |
| **2.0** | **中（デフォルト）** | バランスが良い。十分な aug 効果と自然さ |
| 5.0 | 小（望遠） | ほぼ orthographic。aug 効果が弱い |
| 10.0 | 極小 | ほぼ identity。augmentation として機能しない |

### 残留課題

- 本 EDA は眼中心ピボットのみ。**鼻先ピボット**での再評価が望ましい
- EXP027 の学習済みモデル（バグ修正前に学習）は**再学習が必要**
- 実際の MAE 改善幅は学習してみないとわからない（本指標は幾何特性の proxy）